In [1]:
!pip install torch 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 39.8 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 31.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 36.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 17.4 MB/s  0:00:00
  Attempting uninstall: setuptoolsm━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/10 [sympy]
    Found existing installation: setuptools 82.0.0━━━━━━━━━━━━  2/10 [sympy]
    Uninstalling setuptools-82.0.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/10 [sympy]
      Successfully uninstalled setuptools-82.0.0━━━━━━━━━━━━━━  2/10 [sympy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [torch]m 9/10 [torch]ck]s]


In [2]:
# KL divergence is defined by the difference between two probability distributions
# Defined as  KL(p || q) = H(p,q) - H(p)  where,
#   H(p)   = -sum( p(x) * log(p(x)) )       — entropy of the true distribution
#   H(p,q) = -sum( p(x) * log(q(x)) )       — cross-entropy: observing predicted q(x), sampling from p(x)
#
# Derivation:
# 
#   KL(p || q) = E_{x ~ p}[-log q(x)] - E_{x ~ p}[-log p(x)]
#              = E_{x ~ p}[log p(x) - log q(x)]

#              = E_{x ~ p}[log( p(x) / q(x) )]
#              = sum over x ( p(x) * log( p(x) / q(x) ) )
#
# Property: KL >= 0 always, with equality iff p == q

import torch
import torch.nn.functional as F

p = torch.tensor([0.4, 0.3, 0.2, 0.1])   # true distribution
q = torch.tensor([0.25, 0.25, 0.25, 0.25]) # predicted distribution (uniform)

print(f'p = {p}')
print(f'q = {q}')

p = tensor([0.4000, 0.3000, 0.2000, 0.1000])
q = tensor([0.2500, 0.2500, 0.2500, 0.2500])


In [13]:
# Method  1 derive from cross entropy and entropy defintion 
# KL(p||q) = H(p,q) -  H(p)
#           = - E_(x~p)(log(q(x))) - (-E_(x~p)(log(p(x))))
# entropy is negative log likelihood of a given distribution 
def expectation_p(q,p, eps= 1e-12):
    #p = torch.clip(p, min = 0 , max= 1) 
    # NOTE: This makes sure values are between 0 and 1, but it does not make sure p sums to 1.
    # use torch.clamp instead 
    q = torch.clamp(q, min=eps, max=1.0)
    return - torch.sum(p*torch.log(q))
kl_pq = expectation_p(q,p)- expectation_p(p,p)
print(kl_pq)

# Method 2: KL   is directly calculating from sum log(p(x)/q(x)) p(x)
kl_direct = torch.sum(p * torch.log(p/q))

print("kl_pq", kl_pq)
print("kl_direct", kl_direct )



tensor(0.1064)
kl_pq tensor(0.1064)
kl_direct tensor(0.1064)


In [15]:
# Method 3 PyTorch built-in — F.kl_div expects log(q) as input, p as target
import torch.nn.functional as F
kl_native_pytorch = F.kl_div( torch.log(q), p , reduction = "sum" )

print(f'KL via F.kl_div      = {kl_native_pytorch.item():.6f}')

KL via F.kl_div      = 0.106440


In [ ]:
# questions 
